# AgriEasy Saathi — self-contained fine-tune (no internet needed)

Source embedded below; base model from attached Kaggle Model `qwen-lm/qwen2.5/transformers/1.5b-instruct`. fp16 LoRA on preinstalled libraries only. Output: `/kaggle/working/agrieasy-ai-v1`.

In [ ]:
# 1) stack check (no installs)
import sys, types, importlib.util
_s = importlib.util.spec_from_loader('torchao', loader=None)
_m = types.ModuleType('torchao'); _m.__spec__ = _s; _m.__version__ = '0.19.0'
sys.modules['torchao'] = _m
import torch, transformers, peft, datasets, os, glob, json, base64, pathlib
print('stack ok | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# 2) unpack embedded source + locate attached base model
SRC = json.loads(base64.b64decode("..."))
for p, c in SRC.items():
    pathlib.Path(p).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(p).write_text(c, encoding='utf-8')
print('source unpacked:', len(SRC), 'files')
cands = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'qwen' in p.lower()]
assert cands, 'attached qwen model not found'
BASE = cands[0]
print('base model:', BASE)

In [ ]:
# 3) build dataset
import subprocess
r = subprocess.run('python finetune/build_dataset.py --kb_dir knowledge --seed finetune/seed_qa.jsonl --out finetune/data', shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'dataset build failed'

In [ ]:
# 4) train (fp16 LoRA) — merged model saved to /kaggle/working/agrieasy-ai-v1
r = subprocess.run(f"python finetune/train_lora_plain.py --model_path '{BASE}' --data_dir finetune/data --out /kaggle/working/agrieasy-ai-v1", shell=True)
print('train exit:', r.returncode)
assert r.returncode == 0, 'training failed'

In [ ]:
# 5) smoke test the fine-tuned model
import sys, types, importlib.util
_s = importlib.util.spec_from_loader('torchao', loader=None)
_m = types.ModuleType('torchao'); _m.__spec__ = _s; _m.__version__ = '0.19.0'
sys.modules['torchao'] = _m
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained('/kaggle/working/agrieasy-ai-v1')
model = AutoModelForCausalLM.from_pretrained('/kaggle/working/agrieasy-ai-v1', torch_dtype=torch.float16).to('cuda')
q = 'Meri gehun ki 2 ekad zameen hai, poora plan banayein beej se bikri tak'
ids = tok.apply_chat_template([{'role':'user','content':q}], return_tensors='pt', add_generation_prompt=True).to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=250, temperature=0.5, do_sample=True)
print('=== SMOKE TEST ===')
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))